In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand, when, col, expr
import random

spark = SparkSession.builder.appName("DataGenerator").getOrCreate()

# Sample values
products = ["Laptop", "Mobile", "Tablet", "Watch", "Headphones"]
categories = ["Electronics", "Accessories"]
cities = ["Hyderabad", "Chennai", "Bangalore", "Mumbai", "Delhi", None]

data = []

for i in range(20000):
    order_id = random.randint(1000, 3000)  # duplicates + updates
    customer_id = f"C{random.randint(1, 500)}"
    product = random.choice(products)
    category = random.choice(categories)
    city = random.choice(cities)
    date = f"2024-01-{random.randint(1,28):02d}"
    
    # introduce NULL + negative values
    amount = random.choice([
        random.randint(1000, 60000),
        None,
        -random.randint(1000, 5000)
    ])
    
    quantity = random.randint(1, 5)

    data.append((order_id, customer_id, product, category, city, date, amount, quantity))

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]

df = spark.createDataFrame(data, columns)

df.show(5)

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|    1387|       C324|    Mobile|Accessories|Bangalore|2024-01-15|  5806|       4|
|    1208|       C372|Headphones|Electronics|Hyderabad|2024-01-17| -4435|       2|
|    1079|       C196|     Watch|Accessories|     NULL|2024-01-12| -3455|       1|
|    2981|       C185|Headphones|Accessories|    Delhi|2024-01-24|  2545|       2|
|    1292|       C401|Headphones|Accessories|  Chennai|2024-01-22| -3186|       1|
+--------+-----------+----------+-----------+---------+----------+------+--------+
only showing top 5 rows


In [0]:
df.write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/medallion/flipkart/bronze")

In [0]:
bronze_df = spark.read.option("header", True).csv("/Volumes/workspace/default/medallion/flipkart/bronze")
bronze_df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/medallion/flipkart/bronze/delta")

In [0]:
from pyspark.sql.functions import col, row_number, coalesce, lit
from pyspark.sql.window import Window

silver_df = spark.read.format("delta").load("/Volumes/workspace/default/medallion/flipkart/bronze/delta")

In [0]:
silver_df = silver_df.withColumn("amount", col("amount").cast("int")) \
                     .withColumn("quantity", col("quantity").cast("int")) \
                     .withColumn("date", col("date").cast("date"))

In [0]:
silver_df = silver_df.fillna({
    "city": "Unknown",
    "amount": 0
})

In [0]:
silver_df = silver_df.filter(col("amount") >= 0)

In [0]:
silver_df = silver_df.dropDuplicates()

In [0]:
window_spec = Window.partitionBy("order_id").orderBy(col("date").desc())

silver_df = silver_df.withColumn("rank", row_number().over(window_spec)) \
                     .filter(col("rank") == 1) \
                     .drop("rank")

In [0]:
silver_df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/medallion/flipkart/silver")

In [0]:
gold_df = spark.read.format("delta").load("/Volumes/workspace/default/medallion/flipkart/silver")

Total Sales by Product

In [0]:
sales_by_product = gold_df.groupBy("product") \
    .sum("amount") \
    .withColumnRenamed("sum(amount)", "total_sales")

sales_by_product.show()

+----------+-----------+
|   product|total_sales|
+----------+-----------+
|    Tablet|    6304605|
|    Mobile|    5486642|
|     Watch|    5829073|
|    Laptop|    5765514|
|Headphones|    6714583|
+----------+-----------+



Total Sales by City

In [0]:
sales_by_city = gold_df.groupBy("city") \
    .sum("amount") \
    .withColumnRenamed("sum(amount)", "total_sales")

sales_by_city.show()

+---------+-----------+
|     city|total_sales|
+---------+-----------+
|    Delhi|    4844040|
|  Chennai|    5561987|
|  Unknown|    5476755|
|Hyderabad|    4202682|
|Bangalore|    4847103|
|   Mumbai|    5167850|
+---------+-----------+



Total Revenue

In [0]:
from pyspark.sql.functions import sum as _sum

total_revenue = gold_df.agg(
    _sum("amount").alias("total_revenue")
)

total_revenue.show()

+-------------+
|total_revenue|
+-------------+
|     30100417|
+-------------+



Top Customer

In [0]:
top_customer = gold_df.groupBy("customer_id") \
    .sum("amount") \
    .withColumnRenamed("sum(amount)", "total_spent") \
    .orderBy(col("total_spent").desc()) \
    .limit(1)

top_customer.show()

+-----------+-----------+
|customer_id|total_spent|
+-----------+-----------+
|       C458|     253615|
+-----------+-----------+



Top product

In [0]:
top_product = gold_df.groupBy("product") \
    .sum("amount") \
    .withColumnRenamed("sum(amount)", "total_sales") \
    .orderBy(col("total_sales").desc()) \
    .limit(1)

top_product.show()

+----------+-----------+
|   product|total_sales|
+----------+-----------+
|Headphones|    6714583|
+----------+-----------+



In [0]:
sales_by_product.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/medallion/flipkart/gold/sales_by_product")
sales_by_city.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/medallion/flipkart/gold/sales_by_city")
total_revenue.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/medallion/flipkart/gold/total_revenue")
top_customer.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/medallion/flipkart/gold/top_customer")
top_product.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/medallion/flipkart/gold/top_product")